In [16]:
import pandas as pd

df = pd.read_csv("Dataset/final_phone.csv")

In [ ]:
# models = df['model']
# df = df.drop(columns=['model'])

model_names = df["model"].copy()

df['num_front_cameras'] = pd.to_numeric(df['num_front_cameras'], errors='coerce')
df['num_front_cameras'].fillna(df['num_front_cameras'].mode()[0], inplace=True)
df['num_front_cameras'] = df['num_front_cameras'].astype(int)

df['total_cameras'] = df['num_rear_cameras'] + df['num_front_cameras']

df['price_norm'] = (df['price'] - df['price'].min()) / (df['price'].max() - df['price'].min())
df['ram_norm'] = (df['ram_capacity'] - df['ram_capacity'].min()) / (df['ram_capacity'].max() - df['ram_capacity'].min())

df['camera_score'] = (df['total_cameras'] * 0.4) + (df['price_norm'] * 0.6)
df['gaming_score'] = (df['ram_norm'] * 0.7) + (df['price_norm'] * 0.3)
df['battery_score'] = (df['battery_capacity'] / df['battery_capacity'].max())

In [ ]:
final_features = [
    'gaming_score', 
    'camera_score', 
    'battery_score', 
]

X = df[final_features].copy()

X

,gaming_score,camera_score,battery_score
0,0.020802,1.639345,0.602410
1,0.017179,1.633003,0.602410
2,0.018265,1.634271,0.602410
3,0.015728,1.629197,0.602410
4,0.028413,1.654566,0.602410
...,...,...,...
875,0.018223,0.835542,0.307229
876,0.000428,0.800403,0.313253
877,0.006976,0.813953,0.361446
878,0.008456,0.816008,0.373494


In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
from sklearn.neighbors import NearestNeighbors
knn = NearestNeighbors(metric='cosine',algorithm='brute')
knn.fit(X_scaled)

,n_neighbors,5
,radius,1.0
,algorithm,'brute'
,leaf_size,30
,metric,'cosine'
,p,2
,metric_params,None
,n_jobs,None


In [ ]:
import joblib

joblib.dump(scaler,'scaler.pkl')
joblib.dump(knn,'knn_model.pkl')
joblib.dump(model_names, 'model_names.pkl')
df.to_pickle('phone_data.pkl')

<h2>EXPERIMENTAL PURPOSE</h2>

In [ ]:
bool_cols = ['has_5g','has_nfc','has_ir_blaster']
df[bool_cols] = df[bool_cols].astype(int)

In [ ]:
df['brand_name'].unique()

array(['oneplus', 'samsung', 'motorola', 'realme', 'apple', 'xiaomi',
       'oppo', 'nothing', 'vivo', 'poco', 'iqoo', 'google', 'infinix',
       'tecno', 'lava', 'ikall', 'jio', 'lyf', 'nokia', 'huawei', 'lg',
       'sony', 'micromax', 'itel', 'asus', 'oukitel', 'yu', 'voto',
       'blackberry', 'telefono', 'gionee', 'coolpad', 'easyfone', 'swipe'],
      dtype=object)

In [ ]:
top_brands = df['brand_name'].value_counts().head(10).index

df['brand_name'] = df['brand_name'].where(df['brand_name'].isin(top_brands), 'other')

In [ ]:
def clean_processor(x):
    x = str(x).lower()
    
    if 'snapdragon' in x or 'qualcomm' in x:
        return 'snapdragon'
    elif 'dimensity' in x or 'mediatek' in x:
        return 'mediatek'
    elif 'exynos' in x:
        return 'exynos'
    elif 'bionic' in x or 'a' in x:
        return 'apple'
    elif 'kirin' in x:
        return 'kirin'
    elif 'google' in x:
        return 'google'
    else:
        return 'other'

df['processor_brand'] = df['processor_brand'].apply(clean_processor)

In [ ]:
def clean_os(x):
    x = str(x).lower()
    
    if 'android' in x:
        return 'android'
    elif 'ios' in x:
        return 'ios'
    elif 'hongmeng' in x:
        return 'harmony'
    else:
        return 'other'

df['os'] = df['os'].apply(clean_os)

In [ ]:
df = pd.get_dummies(df, columns=['brand_name','processor_brand','os'])
df.shape

(880, 39)

In [ ]:
df[['res_w','res_h']] = df['resolution'].str.split('x', expand=True).astype(int)
df.drop(columns=['resolution'], inplace=True)

In [ ]:
df

,price,rating,has_5g,has_nfc,has_ir_blaster,num_cores,processor_speed,battery_capacity,fast_charging,ram_capacity,...,processor_brand_kirin,processor_brand_mediatek,processor_brand_other,processor_brand_snapdragon,os_android,os_harmony,os_ios,os_other,res_w,res_h
0,18999,81.0,1,0,0,8.0,2.2,5000.0,33,6.0,...,0,0,0,1,1,0,0,0,1080,2412
1,16499,75.0,1,0,0,8.0,2.4,5000.0,15,4.0,...,0,0,0,0,1,0,0,0,1080,2408
2,16999,80.0,1,1,0,8.0,2.2,5000.0,25,6.0,...,0,0,0,1,1,0,0,0,1080,2408
3,14999,81.0,1,0,0,8.0,2.2,5000.0,0,6.0,...,0,0,0,1,1,0,0,0,1080,2400
4,24999,82.0,1,0,0,8.0,2.6,5000.0,67,6.0,...,0,1,0,0,1,0,0,0,1080,2412
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
875,17500,77.0,0,0,0,8.0,1.5,2550.0,-1,3.0,...,0,0,0,1,1,0,0,0,720,1280
876,3649,77.0,0,0,0,8.0,1.5,2600.0,-1,2.0,...,0,0,0,1,1,0,0,0,1080,1920
877,8990,77.0,0,0,0,8.0,1.5,3000.0,-1,1.0,...,0,0,0,0,1,0,0,0,720,1280
878,9800,77.0,0,0,0,8.0,1.7,3100.0,-1,3.0,...,1,0,0,0,1,0,0,0,1080,1920


In [ ]:

import re

df['screen_size'] = df['screen_size'].apply(
    lambda x: float(re.sub(r'[^0-9.]', '', str(x))) if pd.notnull(x) else x
)

In [ ]:
df['num_front_cameras'] = pd.to_numeric(df['num_front_cameras'], errors='coerce')
df['num_front_cameras'].fillna(df['num_front_cameras'].mode()[0], inplace=True)
df['num_front_cameras'] = df['num_front_cameras'].astype(int)

In [ ]:
df['has_sd_card'] = df['extended_memory'].apply(
    lambda x: 0 if 'not' in str(x).lower() else 1
)

def extract_memory(x):
    match = re.search(r'\d+', str(x))
    return int(match.group()) if match else 0

df['max_expandable'] = df['extended_memory'].apply(extract_memory)

In [ ]:
df.drop(columns=['extended_memory'], inplace=True)

In [ ]:
df.select_dtypes(include='object').columns

Index([], dtype='object')

In [ ]:
df['total_cameras'] = df['num_rear_cameras'] + df['num_front_cameras']

min_p, max_p = df['price'].min(), df['price'].max()
df['price_norm'] = (df['price'] - min_p) / (max_p - min_p)

df['camera_count_norm'] = (
    df['total_cameras'] - df['total_cameras'].min()
) / (
    df['total_cameras'].max() - df['total_cameras'].min()
)

df['camera_score'] = (df['camera_count_norm'] * 0.4) + (df['price_norm'] * 0.6)

df['camera_quality'] = pd.qcut(
    df['camera_score'],
    q=3,
    labels=['Low', 'Medium', 'High']
)

from sklearn.preprocessing import OrdinalEncoder
encoder = OrdinalEncoder(categories=[['Low', 'Medium', 'High']])
df['camera_quality'] = encoder.fit_transform(df[['camera_quality']])

# Normalize RAM
df['ram_norm'] = (
    df['ram_capacity'] - df['ram_capacity'].min()
) / (
    df['ram_capacity'].max() - df['ram_capacity'].min()
)

# Gaming
df['gaming_score'] = (df['ram_norm'] * 0.7) + (df['price_norm'] * 0.3)

# Battery
df['battery_score'] = (df['price_norm'] * 0.6) + (df['ram_norm'] * 0.4)

df['battery_level'] = pd.qcut(df['battery_score'], q=3, labels=['Low', 'Medium', 'High'])

In [ ]:
df

,price,rating,has_5g,has_nfc,has_ir_blaster,num_cores,processor_speed,battery_capacity,fast_charging,ram_capacity,...,max_expandable,total_cameras,price_norm,camera_count_norm,camera_score,camera_quality,ram_norm,gaming_score,battery_score,battery_level
0,18999,81.0,1,0,0,8.0,2.2,5000.0,33,6.0,...,1,4,0.065575,0.5,0.239345,1.0,0.001613,0.020802,0.039990,Medium
1,16499,75.0,1,0,0,8.0,2.4,5000.0,15,4.0,...,1,4,0.055004,0.5,0.233003,1.0,0.000968,0.017179,0.033390,Medium
2,16999,80.0,1,1,0,8.0,2.2,5000.0,25,6.0,...,1,4,0.057118,0.5,0.234271,1.0,0.001613,0.018265,0.034916,Medium
3,14999,81.0,1,0,0,8.0,2.2,5000.0,0,6.0,...,1,4,0.048662,0.5,0.229197,1.0,0.001613,0.015728,0.029843,Medium
4,24999,82.0,1,0,0,8.0,2.6,5000.0,67,6.0,...,0,4,0.090944,0.5,0.254566,1.0,0.001613,0.028413,0.055212,High
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
875,17500,77.0,0,0,0,8.0,1.5,2550.0,-1,3.0,...,128,2,0.059237,0.0,0.035542,0.0,0.000645,0.018223,0.035800,Medium
876,3649,77.0,0,0,0,8.0,1.5,2600.0,-1,2.0,...,32,2,0.000672,0.0,0.000403,0.0,0.000323,0.000428,0.000532,Low
877,8990,77.0,0,0,0,8.0,1.5,3000.0,-1,1.0,...,128,2,0.023255,0.0,0.013953,0.0,0.000000,0.006976,0.013953,Low
878,9800,77.0,0,0,0,8.0,1.7,3100.0,-1,3.0,...,64,2,0.026680,0.0,0.016008,0.0,0.000645,0.008456,0.016266,Low


In [ ]:
from sklearn.preprocessing import StandardScaler

scalar = StandardScaler()

X_scaled = scalar.fit_transform(df)

In [ ]:
X_scaled

array([[-0.1932316 ,  0.65834359,  1.3764944 , ...,  0.25568595,
        -0.70350279,  0.0027851 ],
       [-0.28296841, -0.12578773,  1.3764944 , ...,  0.25568595,
        -0.70350279,  0.0027851 ],
       [-0.26502105,  0.52765503,  1.3764944 , ...,  0.25568595,
        -0.70350279,  0.0027851 ],
       ...,
       [-0.55250192,  0.13558937, -0.72648316, ...,  0.25568595,
        -0.02723584, -1.22266099],
       [-0.52342719,  0.13558937, -0.72648316, ...,  0.25568595,
        -0.36803179, -1.22266099],
       [-0.55250192,  0.13558937, -0.72648316, ...,  0.25568595,
        -0.53842976, -1.22266099]])